In [1]:
import pandas as pd
import re
import unicodedata

# Step 1: Load source datasets
qs = pd.read_csv("2026 QS World University Rankings.csv")
the = pd.read_csv("THE World University Rankings 2016-2026.csv")

# Step 2: Keep only 2026 THE records for temporal consistency
the_2026 = the[the["Year"] == 2026].copy()

# Step 3: Remove duplicates
qs = qs.drop_duplicates()
the_2026 = the_2026.drop_duplicates()

# Step 4: Standardize text values for matching
def strip_accents(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKD", str(value))
    return "".join(ch for ch in value if not unicodedata.combining(ch))

def clean_text(value):
    value = strip_accents(value).lower().strip()
    value = re.sub(r"\([^)]*\)", " ", value)
    value = value.replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()

country_map = {
    "united states of america": "united states",
    "usa": "united states",
    "uk": "united kingdom",
    "china mainland": "china",
    "republic of korea": "south korea",
    "hong kong sar": "hong kong",
    "viet nam": "vietnam",
    "turkiye": "turkey"
}

qs["University_Key"] = qs["Institution Name"].apply(clean_text)
the_2026["University_Key"] = the_2026["Name"].apply(clean_text)
qs["Country_Key"] = qs["Country/Territory"].apply(clean_text).replace(country_map)
the_2026["Country_Key"] = the_2026["Country"].apply(clean_text).replace(country_map)

# Step 5: Convert numeric fields
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.replace("%", "", regex=False),
        errors="coerce"
    )

qs_score_cols = [c for c in qs.columns if "SCORE" in c.upper()]
for col in qs_score_cols:
    qs[col] = to_number(qs[col])

the_numeric_cols = [
    "Rank", "Student Population", "Students to Staff Ratio",
    "Overall Score", "Teaching", "Research Environment",
    "Research Quality", "Industry Impact", "International Outlook"
]
for col in the_numeric_cols:
    the_2026[col] = to_number(the_2026[col])

the_2026["International Students"] = to_number(the_2026["International Students"]) / 100

# Step 6: Merge QS and THE using standardized keys
merged = pd.merge(
    qs,
    the_2026,
    on=["University_Key", "Country_Key"],
    how="inner",
    suffixes=("_QS", "_THE")
)

# Step 7: Normalize ranks so that higher value means better rank
def rank_number(value):
    match = re.search(r"\d+(\.\d+)?", str(value))
    return float(match.group(0)) if match else None

merged["QS_Rank_Number"] = merged["2026 Rank"].apply(rank_number)
merged["THE_Rank_Number"] = merged["Rank"].apply(rank_number)

merged["QS_Rank_Normalized"] = (
    (merged["QS_Rank_Number"].max() - merged["QS_Rank_Number"])
    / (merged["QS_Rank_Number"].max() - merged["QS_Rank_Number"].min())
) * 100

merged["THE_Rank_Normalized"] = (
    (merged["THE_Rank_Number"].max() - merged["THE_Rank_Number"])
    / (merged["THE_Rank_Number"].max() - merged["THE_Rank_Number"].min())
) * 100

# Step 8: Rename and select required columns
final_dataset = merged.rename(columns={
    "2026 Rank": "Rank_QS",
    "Previous Rank": "Previous_Rank",
    "Institution Name": "Name",
    "AR SCORE": "Academic_Reputation_Score",
    "AR RANK": "Academic_Reputation_Rank",
    "ER SCORE": "Employer_Reputation_Score",
    "ER RANK": "Employer_Reputation_Rank",
    "FSR SCORE": "Faculty_Student_Ratio_Score",
    "FSR RANK": "Faculty_Student_Ratio_Rank",
    "CPF SCORE": "Citations_per_Faculty_Score",
    "CPF RANK": "Citations_per_Faculty_Rank",
    "IFR SCORE": "International_Faculty_Score",
    "IFR RANK": "International_Faculty_Rank",
    "ISR SCORE": "International_Student_Score",
    "ISR RANK": "International_Student_Rank",
    "ISD SCORE": "International_Students_Diversity_Score",
    "ISD RANK": "International_Students_Diversity_Rank",
    "IRN SCORE": "International_Research_Network_Score",
    "IRN RANK": "International_Research_Network_Rank",
    "EO SCORE": "Employment_Outcomes_Score",
    "EO RANK": "Employment_Outcomes_Rank",
    "SUS SCORE": "Sustainability_Score",
    "SUS RANK": "Sustainability_Rank",
    "Overall SCORE": "QS_Overall_Score",
    "Rank": "Rank_THE",
    "Student Population": "Student_Population",
    "Students to Staff Ratio": "Students_to_Staff_Ratio",
    "International Students": "International_Students",
    "Female to Male Ratio": "Female_to_Male_Ratio",
    "Overall Score": "THE_Overall_Score",
    "Research Environment": "Research_Environment",
    "Research Quality": "Research_Quality",
    "Industry Impact": "Industry_Impact",
    "International Outlook": "International_Outlook",
    "Country_Key": "Country"
})

final_columns = [
    "Rank_QS", "Previous_Rank", "Name", "Region", "Size", "Focus", "Research", "Status",
    "Academic_Reputation_Score", "Academic_Reputation_Rank",
    "Employer_Reputation_Score", "Employer_Reputation_Rank",
    "Faculty_Student_Ratio_Score", "Faculty_Student_Ratio_Rank",
    "Citations_per_Faculty_Score", "Citations_per_Faculty_Rank",
    "International_Faculty_Score", "International_Faculty_Rank",
    "International_Student_Score", "International_Student_Rank",
    "International_Students_Diversity_Score", "International_Students_Diversity_Rank",
    "International_Research_Network_Score", "International_Research_Network_Rank",
    "Employment_Outcomes_Score", "Employment_Outcomes_Rank",
    "Sustainability_Score", "Sustainability_Rank",
    "QS_Overall_Score", "Rank_THE",
    "Student_Population", "Students_to_Staff_Ratio", "International_Students",
    "Female_to_Male_Ratio", "THE_Overall_Score", "Teaching",
    "Research_Environment", "Research_Quality", "Industry_Impact",
    "International_Outlook", "Year", "Country",
    "QS_Rank_Normalized", "THE_Rank_Normalized"
]

final_dataset = final_dataset[final_columns]

# Step 9: Export final selected-column dataset
final_dataset.to_excel("EduVision_Merged_2026_Selected_Columns.xlsx", index=False)

print("Merged dataset created successfully.")
print("Final shape:", final_dataset.shape)
print("Duplicate universities:", final_dataset.duplicated(subset=["Name", "Country"]).sum())


Merged dataset created successfully.
Final shape: (905, 46)
Duplicate universities: 0
